# Western Visayas Flood Prediction — NB1: Spatial-Temporal Dataset Construction

Builds a **location-day** dataset for machine-learning flood-occurrence prediction across the five
Panay + Guimaras provinces of Western Visayas (Region VI), Philippines.

Each row = one ~5 km grid point on one day, combining:
- **Terrain** (static): elevation, slope, approximate TWI, land cover — from Google Earth Engine
- **Weather** (dynamic): per-point rainfall + antecedent rainfall + typhoon exposure
- **Flood label**: real DSWD DROMIC-recorded flood events (province-level, 2014-2024)

**Data sources:** CHIRPS v2.0 (rainfall), IBTrACS (typhoons), Copernicus GLO-30 DEM + ESA WorldCover
(terrain), GADM v4.1 (boundaries), DSWD DROMIC via ReliefWeb (flood labels).

Author: Claude L. Calimlim - Independent Researcher


## 1. Setup

In [ ]:
!pip install earthengine-api rasterio geopandas rasterstats --quiet

import ee, os, requests, zipfile, warnings
import numpy as np, pandas as pd, geopandas as gpd
import rasterio
from datetime import datetime
from pathlib import Path
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

# --- Earth Engine (requires a registered Cloud project) ---
EE_PROJECT = 'wv-flood-research'   # <-- your Earth Engine project ID
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print('Ready. EE project:', EE_PROJECT)


## 2. Configuration

Edit these paths to match your Google Drive layout.

In [ ]:
BASE      = Path('/content/drive/MyDrive/wv_flood_research')
TIFS_DIR  = BASE / 'chirps_tifs'          # existing CHIRPS daily GeoTIFFs: chirps_wv_YYYYMMDD.tif
OUT_DIR   = BASE
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_START, YEAR_END = 2014, 2024          # DROMIC-labelled window
WET_THRESHOLD        = 1.0                  # mm, defines a 'wet day'
EXPOSURE_RADIUS_KM   = 300                  # typhoon proximity radius
TARGET_PROVS = ['Iloilo', 'Capiz', 'Aklan', 'Antique', 'Guimaras']

tifs = sorted(TIFS_DIR.glob('chirps_wv_*.tif'))
print(f'Found {len(tifs)} CHIRPS TIFs')
if tifs:
    print('First:', tifs[0].name, '| Last:', tifs[-1].name)


## 3. Province boundaries + grid points

We reuse each CHIRPS pixel centre inside the five provinces as a sampling point (native ~5 km
resolution - no interpolation). GADM v4.1 is downloaded automatically if not present.

In [ ]:
GADM_DIR = OUT_DIR / 'gadm'
GADM_DIR.mkdir(exist_ok=True)
GADM_SHP = GADM_DIR / 'gadm41_PHL_1.shp'
if not GADM_SHP.exists():
    print('Downloading GADM v4.1 Philippines provinces...')
    url = 'https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_PHL_shp.zip'
    z = GADM_DIR / 'gadm41_PHL_shp.zip'
    z.write_bytes(requests.get(url, timeout=180).content)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(GADM_DIR)
    print('Extracted.')

gdf = gpd.read_file(GADM_SHP)
namecol = 'NAME_1' if 'NAME_1' in gdf.columns else [c for c in gdf.columns if 'NAME' in c.upper()][0]
gdf_wv = gdf[gdf[namecol].isin(TARGET_PROVS)].copy().to_crs('EPSG:4326')
print('Provinces:', sorted(gdf_wv[namecol].tolist()))

# pixel-centre grid from one TIF
with rasterio.open(tifs[0]) as src:
    H, W = src.height, src.width
    pts = [src.xy(r, c)[::-1] for r in range(H) for c in range(W)]  # (lat, lon)
grid = pd.DataFrame(pts, columns=['lat', 'lon'])

gpts = gpd.GeoDataFrame(grid, geometry=gpd.points_from_xy(grid.lon, grid.lat), crs='EPSG:4326')
joined = gpd.sjoin(gpts, gdf_wv[[namecol, 'geometry']], how='inner', predicate='within')
points = joined[['lat', 'lon', namecol]].rename(columns={namecol: 'province'}).reset_index(drop=True)
print(f'Grid points inside provinces: {len(points)}')
print(points.groupby('province').size())
points.to_csv(OUT_DIR / 'grid_points.csv', index=False)


## 4. Terrain features (Google Earth Engine)

Elevation (Copernicus GLO-30), slope in degrees, an approximate slope-based Topographic Wetness
Index (TWI), and ESA WorldCover land cover. **Note:** TWI here is a slope-based approximation;
a full flow-accumulation TWI (SAGA/Whitebox) is left as future work.

In [ ]:
points = pd.read_csv(OUT_DIR / 'grid_points.csv')

dem = ee.ImageCollection('COPERNICUS/DEM/GLO30').select('DEM').mosaic()
dem30 = dem.reproject(crs='EPSG:4326', scale=30)
slope_deg = ee.Terrain.slope(dem30).rename('slope')          # degrees
slope_rad = slope_deg.multiply(np.pi / 180.0)
tan_slope = slope_rad.tan().max(0.001)
twi = tan_slope.pow(-1).log().rename('TWI')                  # slope-based approximation
lc = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map').rename('landcover')
terrain = (dem.rename('elevation').addBands(slope_deg).addBands(twi).addBands(lc))

def sample_terrain(df, batch=4000, scale=30):
    out = []
    for i in range(0, len(df), batch):
        chunk = df.iloc[i:i+batch]
        feats = [ee.Feature(ee.Geometry.Point([float(r.lon), float(r.lat)]), {'idx': int(r.Index)})
                 for r in chunk.itertuples()]
        samp = terrain.reduceRegions(ee.FeatureCollection(feats), ee.Reducer.mean(), scale)
        for f in samp.getInfo()['features']:
            p = f['properties']
            out.append({'idx': p['idx'], 'elevation': p.get('elevation'), 'slope': p.get('slope'),
                        'TWI': p.get('TWI'), 'landcover': p.get('landcover')})
        print(f'  sampled {min(i+batch, len(df))}/{len(df)}')
    return pd.DataFrame(out).set_index('idx')

points = points.join(sample_terrain(points))
print(points[['elevation', 'slope', 'TWI']].describe())
points.to_csv(OUT_DIR / 'grid_points_terrain.csv', index=False)


## 5. Per-point daily rainfall (fast vectorized sampling)

Samples rainfall at every grid point from the existing CHIRPS TIFs, 2014-2024. Reads each raster
once and fancy-indexes all points - far faster than per-point sampling. CHIRPS nodata (-9999) -> NaN.

In [ ]:
points = pd.read_csv(OUT_DIR / 'grid_points_terrain.csv')

with rasterio.open(tifs[0]) as src:
    inv, H, W = ~src.transform, src.height, src.width
cols, rows = [], []
for lon, lat in zip(points.lon.values, points.lat.values):
    c, r = inv * (lon, lat)
    cols.append(int(round(c - 0.5))); rows.append(int(round(r - 0.5)))
rows = np.clip(np.array(rows), 0, H-1); cols = np.clip(np.array(cols), 0, W-1)
pt_idx = points.index.values

def tif_year(p):
    try: return int(p.stem.replace('chirps_wv_', '')[:4])
    except: return None
keep = [t for t in tifs if (tif_year(t) and YEAR_START <= tif_year(t) <= YEAR_END)]
print(f'Sampling {len(keep)} TIFs at {len(points)} points...')

dates, vals = [], []
for tif in tqdm(keep, desc='TIFs'):
    try: d = datetime.strptime(tif.stem.replace('chirps_wv_', ''), '%Y%m%d').date()
    except ValueError: continue
    with rasterio.open(tif) as src:
        arr = src.read(1)
    v = arr[rows, cols].astype(float); v[v < 0] = np.nan
    dates.append(d); vals.append(v)

mat = np.array(vals)
rain_long = pd.DataFrame({
    'pt_idx': np.tile(pt_idx, len(dates)),
    'date':   np.repeat(pd.to_datetime(dates), len(pt_idx)),
    'rainfall_mm': np.round(mat.ravel(), 4),
})
print('Rows:', rain_long.shape, '| zeros',
      round((rain_long.rainfall_mm == 0).mean()*100, 1), '% | NaN',
      round(rain_long.rainfall_mm.isna().mean()*100, 2), '%')
rain_long.to_parquet(OUT_DIR / 'rain_perpoint_long.parquet')


## 6. Rainfall feature engineering (per point)

Rolling 3-day and 7-day sums, consecutive wet days, month, and season - all computed within each
point's own time series.

In [ ]:
rain_long = pd.read_parquet(OUT_DIR / 'rain_perpoint_long.parquet')

def get_season(m):
    if m in [6, 7, 8, 9]:      return 'habagat'
    if m in [11, 12, 1, 2]:    return 'amihan'
    return 'transition'

def feats(g):
    g = g.sort_values('date').copy()
    r = g['rainfall_mm']
    g['rain_3day_mm'] = r.rolling(3, min_periods=1).sum().round(4)
    g['rain_7day_mm'] = r.rolling(7, min_periods=1).sum().round(4)
    wet = (r > WET_THRESHOLD).astype(int)
    c, run = [], 0
    for w in wet:
        run = run + 1 if w else 0
        c.append(run)
    g['consec_rain_days'] = c
    return g

tqdm.pandas(desc='features')
rain_long = rain_long.groupby('pt_idx', group_keys=False).progress_apply(feats)
rain_long['month']  = rain_long['date'].dt.month
rain_long['season'] = rain_long['month'].map(get_season)
rain_long.to_parquet(OUT_DIR / 'rain_perpoint_features.parquet')
print('Saved features:', rain_long.shape)


## 7. Typhoon exposure per point (IBTrACS)

A point is typhoon-active on a day if a Western-Pacific storm centre passes within 300 km
(haversine from the point itself - finer than a province centroid).

In [ ]:
from math import radians, sin, cos, sqrt, atan2

IBT = OUT_DIR / 'ibtracs_WP.csv'
if not IBT.exists():
    url = ('https://www.ncei.noaa.gov/data/'
           'international-best-track-archive-for-climate-stewardship-ibtracs/'
           'v04r01/access/csv/ibtracs.WP.list.v04r01.csv')
    IBT.write_bytes(requests.get(url, timeout=180).content)

ibt = pd.read_csv(IBT, skiprows=[1], low_memory=False)
ibt['ISO_TIME'] = pd.to_datetime(ibt['ISO_TIME'], errors='coerce')
ibt['LAT'] = pd.to_numeric(ibt['LAT'], errors='coerce')
ibt['LON'] = pd.to_numeric(ibt['LON'], errors='coerce')
ibt = ibt.dropna(subset=['ISO_TIME', 'LAT', 'LON'])
ibt['date'] = pd.to_datetime(ibt['ISO_TIME'].dt.date)
ibt = ibt[(ibt.date.dt.year >= YEAR_START) & (ibt.date.dt.year <= YEAR_END)]

points = pd.read_csv(OUT_DIR / 'grid_points_terrain.csv')
plat, plon = points.lat.values, points.lon.values

def hav(la1, lo1, la2, lo2):
    R = 6371.0; dla = radians(la2-la1); dlo = radians(lo2-lo1)
    a = sin(dla/2)**2 + cos(radians(la1))*cos(radians(la2))*sin(dlo/2)**2
    return R*2*atan2(sqrt(a), sqrt(1-a))

active = {}
for date, grp in tqdm(ibt.groupby('date'), desc='typhoon-days'):
    for slat, slon in zip(grp.LAT.values, grp.LON.values):
        near = (np.abs(plat-slat) < 3.0) & (np.abs(plon-slon) < 3.0)
        for i in np.where(near)[0]:
            if hav(slat, slon, plat[i], plon[i]) <= EXPOSURE_RADIUS_KM:
                active[(i, date)] = 1

typ = pd.DataFrame([(k[0], k[1], 1) for k in active],
                   columns=['pt_idx', 'date', 'typhoon_active'])
typ.to_parquet(OUT_DIR / 'typhoon_perpoint.parquet')
print('Typhoon-active (point,day) pairs:', len(typ))


## 8. Flood labels - DSWD DROMIC events

Flood events are compiled manually from DSWD DROMIC situation reports (ReliefWeb) into
`dromic_flood_events.csv` with columns: `province, date_start, date_end, event, source_url`.
Each event is expanded into daily (province, date) flood-days. Province-level labelling is a
known limitation (a flagged province's points all inherit the flag); disclosed in the paper.

In [ ]:
events = pd.read_csv(OUT_DIR / 'dromic_flood_events.csv')
events['date_start'] = pd.to_datetime(events['date_start'])
events['date_end']   = pd.to_datetime(events['date_end'])
events['province']   = events['province'].str.strip().str.title()

rows = []
for r in events.itertuples():
    for d in pd.date_range(r.date_start, r.date_end, freq='D'):
        rows.append({'province': r.province, 'date': d, 'flood_observed': 1})
flood_days = pd.DataFrame(rows).drop_duplicates(['province', 'date']).reset_index(drop=True)

print(f'Events: {len(events)}  ->  flood-days: {len(flood_days)}')
print(flood_days.groupby('province').size())
flood_days.to_csv(OUT_DIR / 'dromic_flood_days.csv', index=False)


## 9. Assemble final dataset

Join rainfall features + terrain + typhoon + flood label into one training table.

In [ ]:
rain   = pd.read_parquet(OUT_DIR / 'rain_perpoint_features.parquet')
typ    = pd.read_parquet(OUT_DIR / 'typhoon_perpoint.parquet')
points = pd.read_csv(OUT_DIR / 'grid_points_terrain.csv').reset_index().rename(columns={'index': 'pt_idx'})
flood  = pd.read_csv(OUT_DIR / 'dromic_flood_days.csv', parse_dates=['date'])

df = rain.merge(points[['pt_idx', 'lat', 'lon', 'province',
                        'elevation', 'slope', 'TWI', 'landcover']], on='pt_idx', how='left')
df = df.merge(typ, on=['pt_idx', 'date'], how='left')
df['typhoon_active'] = df['typhoon_active'].fillna(0).astype(int)

df['province'] = df['province'].str.title()
flood['province'] = flood['province'].str.title()
df = df.merge(flood[['province', 'date', 'flood_observed']], on=['province', 'date'], how='left')
df['flood_observed'] = df['flood_observed'].fillna(0).astype(int)

pos = df.flood_observed.sum()
print('FINAL:', df.shape, '| positives:', pos, f'({100*pos/len(df):.3f}%)')
print(df.groupby('province')['flood_observed'].sum())
df.to_parquet(OUT_DIR / 'wv_flood_dataset_v2.parquet')
df.to_csv(OUT_DIR / 'wv_flood_dataset_v2.csv', index=False)
print('Saved wv_flood_dataset_v2  ->  feed into NB2')


## 10. Sanity checks

Before modelling: confirm terrain varies, flood days carry more rain than non-flood days, and the
positive rate is plausible.

In [ ]:
df = pd.read_parquet(OUT_DIR / 'wv_flood_dataset_v2.parquet')
print('Terrain spread:\n', df[['elevation', 'slope', 'TWI']].describe(), '\n')
print('Land cover classes:\n', df['landcover'].value_counts(), '\n')
print('Rainfall by label:\n', df.groupby('flood_observed')['rainfall_mm'].describe(), '\n')
print('Class balance:\n', df['flood_observed'].value_counts())
print(f'Positive rate: {100*df.flood_observed.mean():.3f}%')
